Task 6 (LS4): Implement a program which, (a) given one of the feature models and (b) a value k,
– creates (and saves) an image-image similarity matrix,
– performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this image-image
similarity matrix
– stores the latent semantics in a properly named output file
– lists image-weight pairs, ordered in decreasing order of weights

In [19]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [20]:
print("Generating top-", K, " image latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 5  image latent semantics under  layer3  feature space using:  svd


In [21]:
from pathlib import Path
from utils.database_utils import retrieve, compressed_retrieve
from utils.database_utils import store, compressed_store
from feature_models.feature_matrix.image_image_similarity import ImageImageSimilarity

image_feature_vectors_name = f'img_img_sim_{FEATURE_SPACE}.pt'
image_feature_vectors_path = f'./database/{image_feature_vectors_name}'
image_feature_vectors_file = Path(image_feature_vectors_path)

# Check if the image image similarity matrix already present.
if image_feature_vectors_file.is_file():
    image_feature_vectors = compressed_retrieve(image_feature_vectors_name)
else:
    print('Image similiarity matrix doesnot exist, creating one')
    feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')
    image_similarity_generator = ImageImageSimilarity(feature_vectors)
    image_feature_vectors = image_similarity_generator.get_matrix()
    compressed_store(image_feature_vectors, image_feature_vectors_name)

print("Image-Image similarity matrix for image-id 0 as example:\n")
print(image_feature_vectors[0])
print("Shape: ", image_feature_vectors[0][1].shape)


Image similiarity matrix doesnot exist, creating one



 Saving:  img_img_sim_layer3.pt 

Image-Image similarity matrix for image-id 0 as example:

(0, array([1.        , 0.94645673, 0.96963489, ..., 0.80335528, 0.86194342,
       0.83701533]))
Shape:  (4339,)


In [22]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(image_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(image_feature_vectors)

latent_semantics = reducer.reduce_features(image_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[-5.81234836e+01 -1.29448218e+00 -1.04980468e+00 -3.24646781e-01
   2.22273248e-01]
 [-5.78402578e+01 -1.40159763e+00 -7.18405712e-01 -3.20406906e-01
  -1.62832374e-01]
 [-5.83740084e+01 -1.42373779e+00 -8.92252358e-01 -2.88464972e-01
   2.37042543e-01]
 ...
 [-5.35629635e+01  8.24918411e-01 -6.40104458e-01  1.69704489e-01
   5.72635517e-02]
 [-5.70250858e+01  4.02968683e-01 -3.19581738e-01  5.88126227e-01
   3.95828063e-01]
 [-5.64422934e+01  1.10913736e+00 -5.07284459e-01  5.38171589e-01
   1.27165949e-01]]
Shape:  (4339, 5)


In [23]:
store(reducer, f'LS4_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS4_layer3_svd_reducer.pt 



In [24]:
# List image-weight pairs, ordered in decreasing order of weights

# We are to showcase which labels contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_id = [feature_tuple[0] for feature_tuple in image_feature_vectors.values()]

image_weight_tuples = list(zip(image_id, similarity_matrix))

print("Image - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMAGE_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(IMAGE_ID: ", IMAGE_ID, ", Weight: ", weight[i], end="),\t")

Image - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(IMAGE_ID:  7530 , Weight:  -0.012364452823871941),	(IMAGE_ID:  7548 , Weight:  -0.012822454439594146),	(IMAGE_ID:  7524 , Weight:  -0.013067637281177255),	(IMAGE_ID:  7562 , Weight:  -0.013184644661825675),	(IMAGE_ID:  7506 , Weight:  -0.01338035378426745),	(IMAGE_ID:  7928 , Weight:  -0.013385916360768836),	(IMAGE_ID:  7794 , Weight:  -0.01342477025857805),	(IMAGE_ID:  7648 , Weight:  -0.013484685543609938),	(IMAGE_ID:  7540 , Weight:  -0.013492245584312059),	(IMAGE_ID:  7792 , Weight:  -0.013517503681038236),	(IMAGE_ID:  7814 , Weight:  -0.013529228996977123),	(IMAGE_ID:  7908 , Weight:  -0.013530030893657),	(IMAGE_ID:  7780 , Weight:  -0.013549068964062922),	(IMAGE_ID:  7808 , Weight:  -0.013552118679947184),	(IMAGE_ID:  7522 , Weight:  -0.013559978165134936),	(IMAGE_ID:  7766 , Weight:  -0.013608594696596496),	(IMAGE_ID:  5116 , Weight:  -0.013609476763807516),	(IMAGE_ID:  7812